In [1]:
pip install mysql-connector-python pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
import mysql.connector

In [3]:
import mysql.connector
import pandas as pd

In [4]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="kk150725",
    database="retail_project"
)

In [5]:
print(conn.is_connected())

True


In [6]:
df = pd.read_sql("SELECT * FROM clean_retail", conn)

df.shape

C:\Users\USER\anaconda3\lib\site-packages\pandas\io\sql.py:762: UserWarning: pandas only support SQLAlchemy connectable(engine/connection) ordatabase string URI or sqlite3 DBAPI2 connectionother DBAPI2 objects are not tested, please consider using SQLAlchemy
  warnings.warn(


(530100, 9)

In [7]:
df['Sales'] = df['Quantity'] * df['UnitPrice']
df['Sales'].sum()

10666684.540000001

In [8]:
df = df[df['CustomerID'] != 0]
df = df[df['CustomerID'].notna()]

In [9]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

snapshot_date = df['InvoiceDate'].max()

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'Sales': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
,0,132220,1755276.64
12346.0,325,1,77183.60
12347.0,1,182,4310.00
12348.0,74,31,1797.24
12349.0,18,73,1757.55


In [10]:
df['CustomerID'].dtype
df['CustomerID'] = pd.to_numeric(df['CustomerID'], errors='coerce')
df = df[df['CustomerID'].notna()]
df = df[df['CustomerID'] > 0]

In [11]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

snapshot_date = df['InvoiceDate'].max()

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'count',
    'Sales': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
CustomerID,,,
12346.0,325,1,77183.60
12347.0,1,182,4310.00
12348.0,74,31,1797.24
12349.0,18,73,1757.55
12350.0,309,17,334.40


In [12]:
rfm.index.min()

12346.0

In [13]:
def segment(row):
    if row['RFM_Score'] in ['444','443','434','344']:
        return 'Champions'
    
    elif row['F_score'] >= 3 and row['M_score'] >= 3:
        return 'Loyal Customers'
    
    elif row['R_score'] == 4:
        return 'Recent Customers'
    
    elif row['R_score'] <= 2 and row['F_score'] >= 2:
        return 'At Risk'
    
    else:
        return 'Lost'

In [15]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4])

In [16]:
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

In [17]:
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

In [18]:
rfm['Segment'] = rfm.apply(segment, axis=1)

In [19]:
rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Segment
CustomerID,,,,,,,,
12346.0,325,1,77183.60,1,1,4,114,Lost
12347.0,1,182,4310.00,4,4,4,444,Champions
12348.0,74,31,1797.24,2,2,4,224,At Risk
12349.0,18,73,1757.55,3,3,4,334,Loyal Customers
12350.0,309,17,334.40,1,1,2,112,Lost


In [20]:
rfm['Segment'].value_counts()

Lost                1342
Loyal Customers      950
At Risk              857
Champions            839
Recent Customers     350
Name: Segment, dtype: int64

In [21]:
rfm.to_csv("rfm_segments.csv", index=True)

In [22]:
rfm.to_csv("rfm_segments.csv", index=True)

In [23]:
df.to_csv("clean_retail.csv", index=False)

In [24]:
import os
os.listdir()

['.astropy',
 '.conda',
 '.condarc',
 '.continuum',
 '.docker',
 '.gitconfig',
 '.glue',
 '.ipynb_checkpoints',
 '.ipython',
 '.jupyter',
 '.keras',
 '.lesshst',
 '.matplotlib',
 '.spss',
 '.spyder-py3',
 '.vscode',
 '3D Objects',
 'AISWARYA',
 'anaconda3',
 'AppData',
 'Application Data',
 'Bucket FICO scores.ipynb',
 'clean_retail.csv',
 'Contacts',
 'Contract _value.ipynb',
 'Cookies',
 'CreditCode.ipynb',
 'credit_risk_analysis.ipynb',
 'dap',
 'dap project',
 'data analytics quantium',
 'Dmml2fin.ipynb',
 'Documents',
 'Downloads',
 'energy cnspn',
 'Favorites',
 'Final Project',
 'getting-started',
 'IntelGraphicsProfiles',
 'Jedi',
 'Lab 1 Problem 1 Apples and Bananas (1).ipynb',
 'Lab 2 Problem 1 Knapsack Version 1.ipynb',
 'Lab 2 Problem 1 Knapsack Version 2.ipynb',
 'Lect 1 Example 1 (2).ipynb',
 'Lect 1 Example 1 in Pulp (1).ipynb',
 'Links',
 'Local Settings',
 'Microsoft',
 'Music',
 'My Documents',
 'Natural_Gas_Price_Forecast.ipynb',
 'Nat_Gas.csv',
 'NetHood',
 'NTUSER.